# Kairos CIDA v3 LSSDN — Production Training
**Backbone:** Qwen2.5-7B-Instruct (4-bit) | **Datasets:** GSM8K + ARC + BBH + LogicTraps

In [ ]:
!pip install -q torch transformers datasets accelerate bitsandbytes torchdiffeq matplotlib
import getpass, os, sys
token = getpass.getpass("GitHub PAT: ").strip()
if token:
    !git clone https://{token}@github.com/Kairatzh/CIDA-plugin.git
    sys.path.append("CIDA-plugin")
    !pip install -e CIDA-plugin
import torch
print(f"CUDA: {torch.cuda.is_available()}")

In [ ]:
import gc, os, re, json, time, random, math
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from datasets import load_dataset
from tqdm.auto import tqdm

DEVICE = "cuda"
MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"
NUM_PLAN = 8
MAX_SEQ = 384

def flush():
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, padding_side="left")
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16,
                         bnb_4bit_quant_type="nf4", bnb_4bit_use_double_quant=True)
llm = AutoModelForCausalLM.from_pretrained(MODEL_NAME, quantization_config=bnb,
                                            device_map="auto", output_hidden_states=True)
for p in llm.parameters(): p.requires_grad = False
llm.eval()

LLM_HIDDEN = llm.config.hidden_size
embed_tokens = llm.get_input_embeddings()
print(f"LLM: {MODEL_NAME} | Hidden: {LLM_HIDDEN} | VRAM: {torch.cuda.memory_allocated()/1e9:.1f}GB")

In [ ]:
from cida_plugin.config import CIDAPluginConfig
from cida_plugin.core import CIDAPlugin

cfg = CIDAPluginConfig(
    d_input=LLM_HIDDEN, d_hidden=512, d_message=512,
    num_agents=4, num_classes=2, max_rounds=5,
    num_attn_heads=8,
    use_perspective_projector=True, anonymous_messages=True,
    comm_dropout=0.15, early_stop_threshold=None,
    role_prior_blend=0.25,
    # v3 improvements
    sparse_comm_k=2,
    use_dynamic_trust=True, trust_ema_gamma=0.9,
)
cida = CIDAPlugin(cfg).to(DEVICE).float()

class PlanProjectorV3(nn.Module):
    def __init__(self, d_cida, d_llm, num_agents, num_plan=8):
        super().__init__()
        self.num_plan = num_plan; self.d_llm = d_llm
        d_flat = num_agents * d_cida
        self.proj = nn.Sequential(
            nn.Linear(d_flat, d_flat * 2), nn.GELU(), nn.Dropout(0.05),
            nn.Linear(d_flat * 2, num_plan * d_llm),
            nn.LayerNorm(num_plan * d_llm))
    def forward(self, s):
        B = s.size(0)
        return self.proj(s.reshape(B, -1).float()).view(B, self.num_plan, self.d_llm)

proj = PlanProjectorV3(cfg.d_hidden, LLM_HIDDEN, cfg.num_agents, NUM_PLAN).to(DEVICE)

cida_p = sum(p.numel() for p in cida.parameters() if p.requires_grad)
proj_p = sum(p.numel() for p in proj.parameters() if p.requires_grad)
print(f"CIDA: {cida_p/1e6:.2f}M | Proj: {proj_p/1e6:.2f}M | Total: {(cida_p+proj_p)/1e6:.2f}M trainable")

In [ ]:
import random
print("Loading base datasets...")
gsm = load_dataset("openai/gsm8k", "main", split="train")

# 1. PROCEDURAL GENERATOR: RuleTaker & Bongard Logic
def generate_synthetic_logic(num_samples=1000):
    samples = []
    objects = ["glimb", "frop", "snarp", "zool", "blip", "crunk", "plok", "vurp"]
    properties = ["red", "fluffy", "heavy", "invisible", "loud", "spiky", "cold", "shiny"]
    for _ in range(num_samples):
        obj = random.choice(objects)
        prop1, prop2 = random.sample(properties, 2)
        rule_type = random.choice(["direct", "inverse", "negation"])

        if rule_type == "direct":
            rules = f"Rule 1: If a {obj} is {prop1}, then it is {prop2}. Rule 2: A {obj} is {prop1}."
            q = f"Question: Is the {obj} {prop2}?"
            ans = "Yes. Based on Rule 2, the object has the first property, and Rule 1 dictates it must have the second."
        elif rule_type == "inverse":
            rules = f"Rule 1: If a {obj} is {prop1}, then it is NOT {prop2}. Rule 2: A {obj} is {prop1}."
            q = f"Question: Is the {obj} {prop2}?"
            ans = "No. Rule 2 states it has the first property, and Rule 1 states it cannot have the second property."
        else:
            rules = f"Rule 1: Only {prop1} objects are {prop2}. Rule 2: The {obj} is NOT {prop1}."
            q = f"Question: Is the {obj} {prop2}?"
            ans = "No. Since it must be the first property to be the second, and it is not the first, it cannot be the second."

        prompt = f"<|im_start|>system\nApply the rules strictly. Do not use outside knowledge.\n<|im_end|>\n<|im_start|>user\n{rules}\n{q}\n<|im_end|>\n<|im_start|>assistant\n"
        samples.append({"prompt": prompt, "answer": ans, "task": "deduction"})
    return samples

# 2. PROCEDURAL GENERATOR: CLUTRR Relational Induction
def generate_synthetic_relations(num_samples=1000):
    samples = []
    names = ["Alice", "Bob", "Charlie", "Diana", "Eve", "Frank", "Grace", "Hank"]
    for _ in range(num_samples):
        n1, n2, n3 = random.sample(names, 3)
        rel_type = random.choice(["sibling", "parent_child", "grandparent"])

        if rel_type == "sibling":
            rules = f"{n1} is the brother of {n2}. {n2} is the sister of {n3}."
            q = f"What is the relationship between {n1} and {n3}?"
            ans = f"Since {n1} and {n2} are siblings, and {n2} and {n3} are siblings, {n1} is the brother of {n3}."
        elif rel_type == "parent_child":
            rules = f"{n1} is the mother of {n2}. {n2} is the father of {n3}."
            q = f"What is the relationship between {n1} and {n3}?"
            ans = f"{n1} is the grandmother of {n3}, because she is the mother of {n3}'s father."
        else:
            rules = f"{n1} is the grandfather of {n2}. {n3} is the mother of {n2}. {n1} has only one child."
            q = f"What is the relationship between {n1} and {n3}?"
            ans = f"{n1} is the father of {n3}, because {n3} is the parent of {n2} and {n1} is the only grandparent on that side."

        prompt = f"<|im_start|>system\nDeduce the relationship logically.\n<|im_end|>\n<|im_start|>user\n{rules}\n{q}\n<|im_end|>\n<|im_start|>assistant\n"
        samples.append({"prompt": prompt, "answer": ans, "task": "relation"})
    return samples

N_PER_SOURCE = 800

def fmt_gsm(ex):
    q, a = ex["question"], ex["answer"]
    p = f"<|im_start|>system\nSolve step by step.\n<|im_end|>\n<|im_start|>user\n{q}\n<|im_end|>\n<|im_start|>assistant\n"
    return {"prompt": p, "answer": a, "task": "math"}

random.seed(42)
all_samples = []

# 1. Math (Sanity Check)
gsm_fmt = [fmt_gsm(ex) for ex in gsm]
random.shuffle(gsm_fmt)
all_samples.extend(gsm_fmt[:N_PER_SOURCE])

# 2. RuleTaker Procedural Logic
synthetic_logic = generate_synthetic_logic(N_PER_SOURCE)
all_samples.extend(synthetic_logic)

# 3. CLUTRR Procedural Relations
synthetic_relations = generate_synthetic_relations(N_PER_SOURCE)
all_samples.extend(synthetic_relations)

random.shuffle(all_samples)
train_data = all_samples
print(f"Balanced Procedural Dataset: {len(train_data)} samples (Math: {N_PER_SOURCE}, Logic: {N_PER_SOURCE}, Relations: {N_PER_SOURCE})")

In [ ]:
optimizer = torch.optim.AdamW(
    list(cida.parameters()) + list(proj.parameters()), lr=2e-4, weight_decay=0.01)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=len(train_data))

EPOCHS = 1
GRAD_ACCUM = 4
LOG_EVERY = 50
# v3: multi-objective weights
LAMBDA_DISAGREE = 0.05  # penalize high disagreement
LAMBDA_ORTH = 0.02      # encourage diverse agent representations

cida.train(); proj.train()
losses = []; best_loss = float('inf')

for epoch in range(EPOCHS):
    epoch_loss = 0.0
    optimizer.zero_grad()

    for step, sample in enumerate(tqdm(train_data, desc=f"Epoch {epoch+1}")):
        prompt_str, answer_str = sample["prompt"], sample["answer"]
        full_str = prompt_str + answer_str + tokenizer.eos_token

        enc_p = tokenizer(prompt_str, return_tensors="pt", max_length=MAX_SEQ//2, truncation=True).to(DEVICE)
        enc_f = tokenizer(full_str, return_tensors="pt", max_length=MAX_SEQ, truncation=True).to(DEVICE)
        plen = enc_p["input_ids"].shape[1]

        if enc_f["input_ids"].shape[1] <= plen + 1:
            continue

        labels = enc_f["input_ids"].clone()
        labels[:, :plen] = -100

        with torch.no_grad():
            out_p = llm(input_ids=enc_p["input_ids"], attention_mask=enc_p["attention_mask"],
                        output_hidden_states=True, use_cache=False)
            h_last = out_p.hidden_states[-1][0, -1].float().unsqueeze(0)
            full_emb = embed_tokens(enc_f["input_ids"]).float()

        # v3: jitter rounds 2-7 (wider range than v2's 2-5)
        cida.config.max_rounds = int(np.random.randint(2, 8))

        co = cida(pooled=h_last)
        plan = proj(co["state_final"].s)

        p_emb = full_emb[:, :plen].detach()
        a_emb = full_emb[:, plen:].detach()
        cat = torch.cat([p_emb, plan, a_emb], dim=1).to(llm.dtype)

        pm = torch.ones(1, NUM_PLAN, dtype=enc_f["attention_mask"].dtype, device=DEVICE)
        attn = torch.cat([enc_f["attention_mask"][:, :plen], pm, enc_f["attention_mask"][:, plen:]], dim=1)
        shifted = torch.cat([
            torch.full((1, plen + NUM_PLAN), -100, dtype=labels.dtype, device=DEVICE),
            labels[:, plen:]
        ], dim=1)

        outputs = llm(inputs_embeds=cat, attention_mask=attn, labels=shifted)

        # v3 MULTI-OBJECTIVE LOSS (prevents template collapse)
        loss_ce = outputs.loss
        loss_disagree = co["disagreement"].mean() * LAMBDA_DISAGREE
        loss_orth = co["L_orth"] * LAMBDA_ORTH
        loss = (loss_ce + loss_disagree + loss_orth) / GRAD_ACCUM

        loss.backward()

        if (step + 1) % GRAD_ACCUM == 0:
            torch.nn.utils.clip_grad_norm_(list(cida.parameters()) + list(proj.parameters()), 1.0)
            optimizer.step(); scheduler.step(); optimizer.zero_grad()

        raw = loss.item() * GRAD_ACCUM
        epoch_loss += raw; losses.append(raw)

        if (step + 1) % LOG_EVERY == 0:
            avg = epoch_loss / (step + 1)
            vram = torch.cuda.memory_allocated()/1e9
            print(f"  Step {step+1}/{len(train_data)} | Loss: {avg:.4f} | VRAM: {vram:.1f}GB")

        del out_p, h_last, full_emb, co, plan, cat, outputs, loss; flush()

    avg_loss = epoch_loss / len(train_data)
    print(f"Epoch {epoch+1} done. Avg Loss: {avg_loss:.4f}")

    os.makedirs("cida_v3_best", exist_ok=True)
    cfg.save_pretrained("cida_v3_best")
    torch.save(cida.state_dict(), "cida_v3_best/pytorch_model.bin")
    torch.save(proj.state_dict(), "cida_v3_best/plan_projector.pt")
    with open("cida_v3_best/kairos_config.json", "w") as f:
        json.dump({"llm": MODEL_NAME, "num_plan": NUM_PLAN, "loss": avg_loss, "version": "v3-lssdn"}, f, indent=2)
    print("Saved to cida_v3_best/")

In [ ]:
import matplotlib.pyplot as plt
plt.style.use('dark_background')
fig, ax = plt.subplots(figsize=(12, 4))
w = 20
sm = [sum(losses[max(0,i-w):i+1])/min(i+1, w) for i in range(len(losses))]
ax.plot(sm, color='#00d4ff', linewidth=1.5)
ax.set_xlabel('Step'); ax.set_ylabel('Loss')
ax.set_title('CIDA v3 LSSDN Training Loss', fontsize=14, fontweight='bold')
ax.grid(alpha=0.2); plt.tight_layout(); plt.show()

In [ ]:
import shutil
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    dst = "/content/drive/MyDrive/Kairos_Cache/cida_v3_weights"
    os.makedirs(dst, exist_ok=True)
    for f in os.listdir("cida_v3_best"):
        shutil.copy2(f"cida_v3_best/{f}", dst)
    print(f"Saved to Drive: {dst}")
except Exception as e:
    print(f"Drive save failed: {e}")

In [ ]:
cida.eval(); proj.eval()
from IPython.display import display, HTML
import matplotlib.pyplot as plt

@torch.no_grad()
def gen_base(prompt, max_new=150):
    flush()
    enc = tokenizer(prompt, return_tensors="pt", max_length=MAX_SEQ//2, truncation=True).to(DEVICE)
    ids = llm.generate(input_ids=enc["input_ids"], attention_mask=enc["attention_mask"],
        max_new_tokens=max_new, do_sample=False, pad_token_id=tokenizer.pad_token_id)
    return tokenizer.decode(ids[0, enc["input_ids"].size(1):], skip_special_tokens=True)

@torch.no_grad()
def gen_cida(prompt, max_new=150, rounds=5):
    flush()
    enc = tokenizer(prompt, return_tensors="pt", max_length=MAX_SEQ//2, truncation=True).to(DEVICE)
    out = llm(input_ids=enc["input_ids"], attention_mask=enc["attention_mask"],
              output_hidden_states=True, use_cache=False)
    h = out.hidden_states[-1][0, -1].float().unsqueeze(0)
    p_emb = embed_tokens(enc["input_ids"]).detach().float()
    del out; flush()

    cida.config.max_rounds = rounds
    co = cida(pooled=h)
    dis = co["disagreement"].mean().item()

    # v3: trained rollback — if disagreement spikes, retry with perturbation
    if dis > 0.15:
        noise = torch.randn_like(h) * 0.06
        cida.config.max_rounds = rounds + 3
        co = cida(pooled=h + noise)

    plan = proj(co["state_final"].s)
    cat = torch.cat([p_emb, plan], dim=1).to(llm.dtype)
    pm = torch.ones(1, NUM_PLAN, dtype=enc["attention_mask"].dtype, device=DEVICE)
    gm = torch.cat([enc["attention_mask"], pm], dim=1)
    ids = llm.generate(inputs_embeds=cat, attention_mask=gm,
        max_new_tokens=max_new, do_sample=False, pad_token_id=tokenizer.pad_token_id)
    text = tokenizer.decode(ids[0], skip_special_tokens=True)
    return text, co

def chat(q, sys="You are a logical solver. Think step by step."):
    return f"<|im_start|>system\n{sys}\n<|im_end|>\n<|im_start|>user\n{q}\n<|im_end|>\n<|im_start|>assistant\n"

tests = [
    ("Math", "Natalia sold clips to 48 of her friends in April, then half as many in May. How many total?"),
    ("Relational", "Alice has 4 brothers and 3 sisters. How many sisters does Alice's brother have?"),
    ("Trap", "A farmer has 17 sheep. All but 9 die. How many are left?"),
    ("Logic", "If a plane crashes on the border of France and Germany, where do you bury the survivors?"),
    ("Science", "Drop a feather and hammer on the Moon (no atmosphere). Which hits first?"),
    ("Negation", "Rules: 1. If it rains, street is wet. 2. Under a roof, street is dry. 3. It rains, street is under roof. Is street wet?"),
]

print("=== CIDA v3 vs Base Evaluation ===")
for task, q in tests:
    prompt = chat(q)
    b_ans = gen_base(prompt)
    c_ans, co = gen_cida(prompt, rounds=5)
    dis = co["disagreement"].mean().item()

    html = f'''
    <div style="font-family:sans-serif;margin:10px 0;background:#0f172a;padding:14px;border-radius:10px">
      <div style="color:#f8fafc;font-weight:bold;margin-bottom:6px">[{task}] {q}</div>
      <div style="display:grid;grid-template-columns:1fr 1fr;gap:10px">
        <div style="background:#1e293b;border:1px solid #ef4444;border-radius:8px;padding:10px">
          <div style="color:#ef4444;font-weight:bold">Base Qwen 7B</div>
          <div style="color:#e2e8f0;font-family:monospace;white-space:pre-wrap;margin-top:6px;font-size:13px">{b_ans[:300]}</div>
        </div>
        <div style="background:#1e293b;border:1px solid #22d3ee;border-radius:8px;padding:10px">
          <div style="color:#22d3ee;font-weight:bold">CIDA v3 LSSDN (Disagreement: {dis:.4f})</div>
          <div style="color:#e2e8f0;font-family:monospace;white-space:pre-wrap;margin-top:6px;font-size:13px">{c_ans[:300]}</div>
        </div>
      </div>
    </div>'''
    display(HTML(html))

In [ ]:
# Plot belief trajectory for last test
if co and "b_all" in co and len(co["b_all"]) > 0:
    b_all = co["b_all"]
    nr = len(b_all)
    agents = ['Prosecutor', 'Skeptic', 'Defender', 'Integrator']
    colors = ['#dc3545', '#ffc107', '#28a745', '#17a2b8']
    plt.figure(figsize=(10, 5))
    plt.style.use('dark_background')
    for a in range(min(4, b_all[0].size(1))):
        vals = [torch.sigmoid(b_all[r][0, a, -1]).item() for r in range(nr)]
        plt.plot(range(1, nr+1), vals, marker='o', label=agents[a], color=colors[a], linewidth=2, markersize=6)
    plt.title("CIDA v3 Agent Deliberation Trajectory", fontsize=13, fontweight='bold', color='#f8fafc')
    plt.xlabel("Deliberation Round"); plt.ylabel("Belief State")
    plt.xticks(range(1, nr+1)); plt.grid(True, alpha=0.15); plt.legend(); plt.tight_layout(); plt.show()